# Benchmark curation for the manuscript v5 repository

This local assembly record copies checksum-registered benchmark evidence into the paper repository, updates only the portable notebooks’ environment records, and executes the result and evidence notebooks. Historical source and the original workspace remain unchanged. This curator needs the original MacBook workspace; it is provenance rather than a clean-checkout analysis entry point.

Publication note: author path labels were normalized. Saved numerical outputs record the earlier execution. This source record needs the author archives; use the current portable evidence and preservation notebooks for checkout-level verification.

In [1]:
from pathlib import Path
import hashlib, importlib.metadata, json, platform, sys
import numpy as np
import pandas as pd
from IPython.display import display

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1048576), b''):
            digest.update(block)
    return digest.hexdigest()

HISTORICAL_MAC_ENVIRONMENT = {
    'manager': 'existing micromamba-compatible environment, reused without modification',
    'name': 'chronoclone',
    'absolute_macbook_path': '/path/to/ChronoClone/.runtime/chronoclone',
    'kernel': 'chronoclone',
}
runtime = dict(HISTORICAL_MAC_ENVIRONMENT,
    current_prefix=sys.prefix, current_executable=sys.executable, python=platform.python_version(),
    versions={name: importlib.metadata.version(name) for name in
              ['numpy','pandas','scipy','scikit-learn','scanpy','matplotlib','nbformat','nbclient']})

PROJECT = Path('/path/to/ChronoClone')
REPOSITORY = PROJECT / 'analysis_workspace/ChronoClone_paper'
BENCHMARKS = REPOSITORY / 'benchmarks'
CURATION = BENCHMARKS / 'curation'
plan = json.loads((CURATION/'source_copy_plan.json').read_text())
runtime['execution_command'] = '/path/to/ChronoClone/.runtime/chronoclone/bin/python analysis_workspace/workspace_audits/20260914_paper_v5/execute_notebook.py analysis_workspace/ChronoClone_paper/benchmarks/curation/0001_curate_benchmark.ipynb'
assert Path(sys.prefix).resolve() == Path(HISTORICAL_MAC_ENVIRONMENT['absolute_macbook_path']).resolve()
display(pd.DataFrame([runtime]).T)


,0
manager,"existing micromamba-compatible environment, re..."
name,chronoclone
absolute_macbook_path,~/.reference-science/conda/envs/...
kernel,chronoclone
current_prefix,~/.reference-science/conda/envs/...
current_executable,~/.reference-science/conda/envs/...
python,3.11.15
versions,"{'numpy': '2.4.6', 'pandas': '2.3.3', 'scipy':..."
execution_command,~/.reference-science/conda/envs/...


## Copy only registered sources

In [2]:
import shutil
copy_records=[]
for record in plan['copy_files']:
    source=PROJECT/record['source']
    destination=REPOSITORY/record['destination']
    assert destination.resolve().is_relative_to(BENCHMARKS.resolve())
    assert source.stat().st_size==record['bytes'] and sha256(source)==record['sha256'], source
    destination.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(source,destination)
    assert sha256(destination)==record['sha256'], destination
    copy_records.append(dict(record,destination_sha256_at_copy=sha256(destination)))
(CURATION/'copy_manifest.json').write_text(json.dumps(copy_records,indent=2)+'\n')
print('Exact source copies:',len(copy_records))
print('Copied bytes:',sum(record['bytes'] for record in copy_records))


Exact source copies: 94
Copied bytes: 55797133


## Adapt and execute the portable notebooks

Only the runtime record and execution command change. The scientific calculation and plot cells remain intact. The source checksum and resulting notebook checksum are recorded separately.

In [3]:
import nbformat, subprocess
notebooks=BENCHMARKS/'ode_noise/notebooks'
adapted=[]
for name in ['0003_t1_locked_method_noise_results.ipynb','0005_t1_locked_four_method_noise_results.ipynb']:
    path=notebooks/name
    notebook=nbformat.read(path,as_version=4)
    source_hash=sha256(path)
    setup=next(cell for cell in notebook.cells if cell.cell_type=='code' and 'ENVIRONMENT = {' in cell.source)
    setup.source=setup.source.replace('"name": "chronoclone-paper"','"name": "chronoclone"')
    setup.source=setup.source.replace('"path": "see environment.yml"','"historical_macbook_path": "/path/to/ChronoClone/.runtime/chronoclone",\n    "current_prefix": __import__("sys").prefix')
    setup.source=setup.source.replace('"kernel": "python3"','"kernel": "chronoclone"')
    setup.source=setup.source.replace('python ../../scripts/execute_notebook_inprocess.py','python ../run_notebook.py')
    notebook.metadata.kernelspec={'name':'chronoclone','display_name':'chronoclone','language':'python'}
    for cell in notebook.cells:
        if cell.cell_type=='code': cell.execution_count=None;cell.outputs=[]
    nbformat.write(notebook,path)
    adapted.append({'path':str(path.relative_to(REPOSITORY)),'source_sha256':source_hash})
for name in ['0003_t1_locked_method_noise_results.ipynb','0005_t1_locked_four_method_noise_results.ipynb','0006_benchmark_evidence_and_smoke.ipynb']:
    command=[sys.executable,str(BENCHMARKS/'run_notebook.py'),str(notebooks/name),'--kernel','chronoclone']
    completed=subprocess.run(command,capture_output=True,text=True)
    if completed.returncode: raise RuntimeError(completed.stdout+'\n'+completed.stderr)
    print(completed.stdout.strip())
for record in adapted:
    record['executed_sha256']=sha256(REPOSITORY/record['path'])
(CURATION/'adapted_notebooks.json').write_text(json.dumps(adapted,indent=2)+'\n')


Executed /path/to/ChronoClone/ChronoClone_paper/benchmarks/ode_noise/notebooks/0003_t1_locked_method_noise_results.ipynb: 20 code cells


Executed /path/to/ChronoClone/ChronoClone_paper/benchmarks/ode_noise/notebooks/0005_t1_locked_four_method_noise_results.ipynb: 19 code cells


Executed /path/to/ChronoClone/ChronoClone_paper/benchmarks/ode_noise/notebooks/0006_benchmark_evidence_and_smoke.ipynb: 5 code cells


560

## Verify preservation and record execution

In [4]:
errors=[]
for record in copy_records:
    source=PROJECT/record['source']
    assert sha256(source)==record['sha256'], source
    destination=REPOSITORY/record['destination']
    record['destination_sha256_after_execution']=sha256(destination)
    record['destination_bytes_after_execution']=destination.stat().st_size
    if record['destination'].startswith('benchmarks/frozen_runtime/'):
        assert record['destination_sha256_after_execution']==record['sha256'],destination
for name in ['0003_t1_locked_method_noise_results.ipynb','0005_t1_locked_four_method_noise_results.ipynb','0006_benchmark_evidence_and_smoke.ipynb']:
    nb=nbformat.read(notebooks/name,as_version=4)
    cells=[c for c in nb.cells if c.cell_type=='code']
    assert [c.execution_count for c in cells]==list(range(1,len(cells)+1))
    assert not [o for c in cells for o in c.outputs if o.output_type=='error']
(CURATION/'copy_manifest.json').write_text(json.dumps(copy_records,indent=2)+'\n')
result=json.loads((BENCHMARKS/'ode_noise/reports/benchmark_evidence_validation.json').read_text())
assert result['status']=='pass'
print('Scientific cores and wheel remain byte-preserved; ancillary publication path labels are adapted.')
display(pd.DataFrame([{k:v for k,v in result.items() if k!='runtime'}]).T)


Scientific cores and wheel remain byte-preserved; ancillary publication path labels are adapted.


,0
schema,chronoclone_paper_benchmark_evidence_v1
status,pass
summary_rows,13005
grid_coordinates,2601
historical_method_condition_rows,1300500
historical_method_failures,248166
summary_hashes_verified,7
bounded_smoke_status,pass
full_aggregate_verified_here,False
full_benchmark_rerun,False
